# FER2013 — არქიტექტურა 1: TinyCNN

მინიმალური CNN — განზრახ underfitting-ის საბაზო ხაზი.

| Run | Optimizer | LR |
|---|---|---|
| arch1_adam_lr1e3 | Adam | 1e-3 |
| arch1_adam_lr1e4 | Adam | 1e-4 |
| arch1_sgd_lr1e2 | SGD | 1e-2 |

## დაყენება და იმპორტი

In [ ]:
!pip install -q wandb

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report

import wandb

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)

## WandB კავშირი

In [ ]:
from kaggle_secrets import UserSecretsClient
wandb.login(key=UserSecretsClient().get_secret('WANDB_API_KEY'))

## მონაცემების ჩატვირთვა

In [ ]:
CSV_PATH = '/kaggle/input/challenges-in-representation-learning-facial-expression-recognition-challenge/fer2013.csv'
EMOTION_LABELS = {0:'Angry', 1:'Disgust', 2:'Fear', 3:'Happy', 4:'Sad', 5:'Surprise', 6:'Neutral'}

df = pd.read_csv(CSV_PATH)
print(df['Usage'].value_counts())

In [ ]:
train_df = df[df['Usage'] == 'Training']
counts   = train_df['emotion'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.bar([EMOTION_LABELS[i] for i in counts.index], counts.values, color='steelblue')
ax.bar_label(bars)
ax.set_title('Training Set — Class Distribution')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 7, figsize=(14, 2.5))
for eid in range(7):
    sample = train_df[train_df['emotion'] == eid].iloc[0]
    axes[eid].imshow(np.array(sample['pixels'].split(), dtype=np.uint8).reshape(48, 48), cmap='gray')
    axes[eid].set_title(EMOTION_LABELS[eid], fontsize=9)
    axes[eid].axis('off')
plt.tight_layout()
plt.show()

## Dataset და DataLoader

In [ ]:
class FER2013Dataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.pixels    = dataframe['pixels'].tolist()
        self.labels    = dataframe['emotion'].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = Image.fromarray(
            np.array(self.pixels[idx].split(), dtype=np.uint8).reshape(48, 48), mode='L'
        )
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.labels[idx], dtype=torch.long)


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_df = df[df['Usage'] == 'Training'].reset_index(drop=True)
val_df   = df[df['Usage'] == 'PublicTest'].reset_index(drop=True)
test_df  = df[df['Usage'] == 'PrivateTest'].reset_index(drop=True)

BATCH_SIZE   = 64
train_loader = DataLoader(FER2013Dataset(train_df, transform), BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(FER2013Dataset(val_df,   transform), BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')

## არქიტექტურა 1: TinyCNN

```
Input (B, 1, 48, 48)
  Conv(1→8)  → ReLU → MaxPool   (B, 8,  24, 24)
  Conv(8→16) → ReLU → MaxPool   (B, 16, 12, 12)
  Flatten                        (B, 2304)
  Linear(2304→128) → ReLU
  Linear(128→7)
```

მხოლოდ 2 conv layer, 8/16 ფილტრი, BatchNorm არ არის — მოდელი განზრახ underfitted.

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.conv1 = nn.Conv2d(1,  8,  kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8,  16, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(16 * 12 * 12, 128)
        self.fc2   = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


model = TinyCNN().to(DEVICE)
print(model)
print(f'პარამეტრები: {sum(p.numel() for p in model.parameters()):,}')

## Forward Pass შემოწმება

In [ ]:
model.eval()
dummy = torch.randn(4, 1, 48, 48).to(DEVICE)

with torch.no_grad():
    out = model(dummy)

print(f'Input:  {dummy.shape}')
print(f'Output: {out.shape}')
print(f'NaN: {torch.isnan(out).any().item()} | Inf: {torch.isinf(out).any().item()}')

assert out.shape == (4, 7)
assert not torch.isnan(out).any()
print('Forward pass OK')

## Backward Pass შემოწმება

In [ ]:
model.train()
model.zero_grad()

dummy_labels = torch.randint(0, 7, (4,)).to(DEVICE)
loss = nn.CrossEntropyLoss()(model(dummy), dummy_labels)
loss.backward()

print(f'{'Layer':<20} {'Mean |grad|':>14} {'Max |grad|':>12}')
print('-' * 48)
names, avg_g, max_g = [], [], []
for name, p in model.named_parameters():
    if p.requires_grad and p.grad is not None:
        m, mx = p.grad.abs().mean().item(), p.grad.abs().max().item()
        print(f'{name:<20} {m:>14.2e} {mx:>12.2e}')
        names.append(name); avg_g.append(m); max_g.append(mx)

fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(range(len(avg_g)), max_g, alpha=0.4, color='steelblue', label='max')
ax.bar(range(len(avg_g)), avg_g, alpha=0.9, color='coral',     label='mean')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=40, ha='right', fontsize=8)
ax.set_title('Gradient Flow — TinyCNN')
ax.legend()
plt.tight_layout()
plt.show()

model.zero_grad()
print('Backward pass OK')

## სწავლების ციკლი

In [ ]:
def compute_grad_norm(model):
    return sum(
        p.grad.detach().norm(2).item() ** 2
        for p in model.parameters() if p.grad is not None
    ) ** 0.5


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total, gn = 0.0, 0, 0, 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        gn = compute_grad_norm(model)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total, gn


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out  = model(imgs)
        loss = criterion(out, labels)
        total_loss += loss.item() * imgs.size(0)
        preds       = out.argmax(1)
        correct    += (preds == labels).sum().item()
        total      += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels


def run_experiment(run_name, config):
    run = wandb.init(
        project='fer2013',
        name=run_name,
        config=config,
        tags=['architecture_1', 'tiny_cnn'],
        reinit=True
    )

    model     = TinyCNN().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = (
        torch.optim.Adam(model.parameters(), lr=config['lr'])
        if config['optimizer'] == 'adam'
        else torch.optim.SGD(model.parameters(), lr=config['lr'], momentum=0.9)
    )

    best_val_acc = 0.0
    for epoch in range(1, config['epochs'] + 1):
        tr_loss, tr_acc, gn = train_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc, preds, labels = evaluate(model, val_loader, criterion)

        wandb.log({
            'epoch': epoch,
            'train/loss': tr_loss, 'train/accuracy': tr_acc,
            'val/loss':   vl_loss, 'val/accuracy':   vl_acc,
            'grad_norm':  gn,
        }, step=epoch)

        if epoch % 5 == 0 or epoch == 1:
            print(f'Epoch {epoch:2d} | train={tr_acc:.3f} val={vl_acc:.3f} gap={tr_acc-vl_acc:+.3f} | gn={gn:.3f}')

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(model.state_dict(), f'{run_name}_best.pt')

    cm = confusion_matrix(labels, preds, labels=list(range(7)))
    wandb.log({'confusion_matrix': wandb.Table(
        columns=['True \\ Pred'] + list(EMOTION_LABELS.values()),
        data=[[EMOTION_LABELS[i]] + r.tolist() for i, r in enumerate(cm)]
    )})
    wandb.summary['best_val_accuracy'] = best_val_acc
    run.finish()
    print(f'Best val acc: {best_val_acc:.4f}\n')
    return best_val_acc, model

## ექსპერიმენტი 1 — Adam, LR=1e-3

In [ ]:
acc1, model1 = run_experiment('arch1_adam_lr1e3', {
    'architecture': 'tiny_cnn', 'optimizer': 'adam',
    'lr': 1e-3, 'batch_size': 64, 'epochs': 30,
    'augmentation': False, 'dropout': 0.0
})

## ექსპერიმენტი 2 — Adam, LR=1e-4

In [ ]:
acc2, model2 = run_experiment('arch1_adam_lr1e4', {
    'architecture': 'tiny_cnn', 'optimizer': 'adam',
    'lr': 1e-4, 'batch_size': 64, 'epochs': 30,
    'augmentation': False, 'dropout': 0.0
})

## ექსპერიმენტი 3 — SGD, LR=1e-2

In [ ]:
acc3, model3 = run_experiment('arch1_sgd_lr1e2', {
    'architecture': 'tiny_cnn', 'optimizer': 'sgd',
    'lr': 1e-2, 'batch_size': 64, 'epochs': 30,
    'augmentation': False, 'dropout': 0.0
})

## შედეგები და ანალიზი

In [ ]:
results = {'arch1_adam_lr1e3': acc1, 'arch1_adam_lr1e4': acc2, 'arch1_sgd_lr1e2': acc3}

for name, acc in results.items():
    print(f'{name:<22} {acc:.4f}')

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(results.keys(), results.values(), color='steelblue')
ax.set_ylabel('Best Val Accuracy')
ax.set_title('Architecture 1 — TinyCNN Results')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
best_model = max([(acc1, model1), (acc2, model2), (acc3, model3)], key=lambda x: x[0])[1]
best_model.eval()

imgs_batch, lbls_batch = next(iter(val_loader))
with torch.no_grad():
    pred_batch = best_model(imgs_batch.to(DEVICE)).argmax(1).cpu()

fig, axes = plt.subplots(3, 5, figsize=(12, 7))
for i, ax in enumerate(axes.flat):
    img = (imgs_batch[i].squeeze().numpy() * 0.5 + 0.5).clip(0, 1)
    ax.imshow(img, cmap='gray')
    color = 'green' if lbls_batch[i] == pred_batch[i] else 'red'
    ax.set_title(f'{EMOTION_LABELS[lbls_batch[i].item()]} / {EMOTION_LABELS[pred_batch[i].item()]}',
                 fontsize=7, color=color)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
_, _, final_preds, final_labels = evaluate(best_model, val_loader, nn.CrossEntropyLoss())
print(classification_report(final_labels, final_preds, target_names=list(EMOTION_LABELS.values())))